# Aegis-Her — verified Spark-X2.6-4B evidence
Choose a T4 GPU and run all cells. Thinking is disabled so the model returns structured JSON rather than a reasoning trace.


In [ ]:
!pip -q install transformers==4.57.1 accelerate sentencepiece


In [ ]:
import json, torch, transformers, time
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_ID='XHToken/Spark-X2.6-4B'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,trust_remote_code=True,dtype='auto',device_map='auto')
print('Loaded',MODEL_ID,'on',model.device)


In [ ]:
notice='社区诊所通知：孕妇若出现严重头痛、视力模糊、上腹疼痛或面部突然肿胀，应立即联系急诊服务。预约咨询电话 [PHONE-REDACTED]，患者身份证 [ID-REDACTED]。'
schema={'summary':'','urgent_actions':[],'next_steps':[],'documents':[],'deadlines':[],'uncertainties':[],'human_review':''}
prompt=f'''You are Aegis-Her, a cautious public-information accessibility assistant. Respond in Simplified Chinese. Use only facts in SOURCE. Never diagnose, determine eligibility, invent deadlines, rank people, or reveal reasoning. Return JSON only with exactly this schema:
{json.dumps(schema,ensure_ascii=False)}
SOURCE:
{notice}
JSON:'''
messages=[{'role':'user','content':prompt}]
inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,enable_thinking=False,return_dict=True,return_tensors='pt').to(model.device)
started=time.time()
with torch.inference_mode(): outputs=model.generate(**inputs,max_new_tokens=600,do_sample=False)
raw=tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True)
start,end=raw.find('{'),raw.rfind('}')
parsed=json.loads(raw[start:end+1])
evidence={'model':MODEL_ID,'model_commit':getattr(model.config,'_commit_hash',None),'transformers':transformers.__version__,'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU','dtype':str(model.dtype),'thinking_enabled':False,'max_new_tokens':600,'do_sample':False,'latency_seconds':round(time.time()-started,2),'sanitized_input':notice,'actual_output':parsed,'raw_unedited_output':raw}
print(json.dumps(evidence,ensure_ascii=False,indent=2))
with open('aegis_her_evidence.json','w',encoding='utf-8') as f: json.dump(evidence,f,ensure_ascii=False,indent=2)
